**MULTI CLASSIFICATION NEURAL NETWORK**(From Scratch using Numpy):

Here , In this I have a trained a  Multi Classification neural network from scratch using raw python code numpy library.


No deep learning frameworks(tensorflow,pytorch etc) used .

It is similiar like binary classificaton but the only thing here we use one hot encoding to label the multiple classes so that the output probabilities pedict right one.

For multi classification we use softmax activation function to divide the probabilities of output layer .

The flow of a Neural Network training:(Similar)

DATA  ->  SPLITING(Features,Target)  ->  SCALING DATA  ->  WEIGHTS AND BIAS  ->  FORWARD PROPAGATION  ->  LOSS  ->  BACKWARD PROPAGATION  ->  UPDATE WEIGHTS AND BIAS  ->  **TEST** MODEL

In [267]:
!pip install numpy scikit-learn

Here , we take the wine dataset from sklearn.

The data consist of 178 samples and 13 features.[X(178,13)]. The target is to  find the type of wine based on the features.


class 0 → Wine type1

class 1 → Wine type2

class 2 → Wine type3


In [268]:
import numpy as np
from sklearn.datasets import load_wine


data = load_wine()
X = data.data
y = data.target

print(X.shape)
print(y.shape)





(178, 13)
(178,)


Spliting the data for 80% for training and 20% for testing the data.


After output classes should be one hot encoded and labeled as classes.

eye() function in numpy used for one hot encoding.

In [269]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

def one_hot_encode(y,num_classes=3):
   return np.eye(num_classes)[y]

y_train = one_hot_encode(y_train)
y_test = one_hot_encode(y_test)





We have scaled the values of the features . On scaling the computation makes simple and more accurate  with out scaling .

In [270]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

 n_x -> Sample size 178 rows


 n_h1 -> Hidden layer1 Neurons

 n_h2 -> HIdden layer2 Neurons

 n_y -> Output Layer  Neurons (num_classes)

 Weights & Bias:

 w1 and b1 ->  for Input Layer


 w2 and b2 -> for Hidden Layer1

 w3 and b3 -> for HIdden Layer2

In [271]:
def intial_parameters(n_x,n_h1,n_h2,n_y):
  w1 = np.random.randn(n_x,n_h1)*0.01
  b1 = np.zeros((1,n_h1))

  w2 = np.random.randn(n_h1,n_h2)*0.01
  b2 = np.zeros((1,n_h2))

  w3 = np.random.randn(n_h2,n_y)*0.01
  b3 = np.zeros((1,n_y))


  return w1,b1,w2,b2,w3,b3


Activation Functions:
 these mattes more in a neural network training it is introduce to learn complex patterns in the data . Introdues non-linearity.

 For multi classification we use softmax activation function

 Softmax : output neuron
 Relu : Hidden Layers

In [272]:
def softmax(z):
  exp_z = np.exp(z)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def Relu(z):
  return np.maximum(0,z)


Forward pass of the neural network here where we train the neural network .
  z=w⋅x+b (mathematical representation)

  z1 -> a1=Relu(z1) -> z2 -> a2=Relu(z2) -> z3 -> a3=softmax(z3)

In [273]:
def forward_propagation(x,w1,b1,w2,b2,w3,b3):
  z1 = np.dot(x,w1)+b1
  a1 = Relu(z1)

  z2 = np.dot(a1,w2)+b2
  a2 = Relu(z2)

  z3 = np.dot(a2,w3)+b3
  a3 = softmax(z3)

  return z1, a1, z2, a2, z3, a3

For Multi Classificaton we use Categorical Cross Entropy loss function to calculate the loss . The loss function shows us how much error the model makes so based on that we can update the weights and bias .

In [274]:
def loss_function(y,y_pred):
  m=y.shape[0]
  loss = -np.sum(y*np.log(y_pred + 1e-8))/m
  return loss

Backward Propagation : This works on the concept Gradient Descent . It calculate the deriavative of the loss and updates the weights and bias .

NOTE:

->The output layer gradient is propostional to how much prediction differs from true label.

-> input*error=gradient

Here the extra content is deriavative of relu function because of we have 2 hidden layers we ned to also calculate gradients for hidden layers also.


In [275]:

def relu_deriavative(z):
  return (z>0).astype(float)


def backward_propagation(x, y, a1, a2, a3, w2, w3, z1, z2):
    m = x.shape[0]

    dz3 = a3 - y
    dw3 = np.dot(a2.T, dz3)
    db3 = np.sum(dz3, axis=0, keepdims=True)/m

    da2 = np.dot(dz3, w3.T)
    dz2 = da2*relu_deriavative(z2)
    dw2 = (1/m)*np.dot(a1.T,dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)/m

    da1 = np.dot(dz2, w2.T)
    dz1 = da1*relu_deriavative(z1)
    dw1 = (1/m)*np.dot(x.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)/m

    return dw1, db1, dw2, db2, dw3, db3



Learning rate & Updating of weights and bias:

 -> learning rate is for how much step size the model moving for optimal solution.

 ->it updates the model parameters during the backpropagation.

 ->small learning rate -> better solution


In [276]:
def upgrade_parameters(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, learning_rate):

    w1 = w1 - learning_rate*dw1
    b1 = b1 - learning_rate*db1

    w2 = w2 - learning_rate*dw2
    b2 = b2 - learning_rate*db2

    w3 = w3 - learning_rate*dw3
    b3 = b3 - learning_rate*db3

    return w1, b1, w2, b2, w3, b3

Here , is the training loop starts we train the model for number itereations so that the model learns patterns more effective and decreses the loss function.

We update the weights and bias for number of epochs and uses the best model .

In [277]:
def train(x,y,n_h1,n_h2,epochs,learning_rate):
  n_x=x.shape[1]
  n_y=3

  w1,b1,w2,b2,w3,b3 = intial_parameters(n_x,n_h1,n_h2,n_y)

  for i in range(epochs):
    z1,a1,z2,a2,z3,a3 = forward_propagation(x,w1,b1,w2,b2,w3,b3)
    loss = loss_function(y,a3)
    dw1, db1, dw2, db2 ,dw3, db3 = backward_propagation(x,y, a1, a2, a3, w2, w3, z1,z2)
    w1, b1, w2, b2 , w3, b3 = upgrade_parameters(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, learning_rate)
    if i % 100 == 0:
        print("Epoch: ", i, "Loss: ", loss)

  return w1,b1,w2,b2,w3,b3


Predict the output which type of wine it is.

In [278]:
def predict(x,w1,b1,w2,b2,w3,b3):
  z1,a1,z2,a2,z3,a3 = forward_propagation(x,w1,b1,w2,b2,w3,b3)
  return np.argmax(a3, axis=1)

Test Accuracy of the model

-> We observe the model on training the for number of iteration decreses the loss function and increase the acuuracy

In [279]:
np.random.seed(42)

Test on train data set

In [280]:
w1,b1,w2,b2 ,w3,b3= train(x_train,y_train, n_h1=9,n_h2=6, epochs=1500,learning_rate=0.01)
y_pred = predict(x_train,w1,b1,w2,b2,w3,b3)
y_pred = one_hot_encode(y_pred)
acuuracy = np.mean(y_pred == y_train)
print(acuuracy)

Epoch:  0 Loss:  1.098611127407084
Epoch:  100 Loss:  1.0930670178809554
Epoch:  200 Loss:  1.0902253609880515
Epoch:  300 Loss:  1.0887457561604827
Epoch:  400 Loss:  1.0878987742767166
Epoch:  500 Loss:  1.0871877215632668
Epoch:  600 Loss:  1.085669024212917
Epoch:  700 Loss:  1.0758455484311822
Epoch:  800 Loss:  0.8991323118503657
Epoch:  900 Loss:  0.3048651050930073
Epoch:  1000 Loss:  0.09854745637085933
Epoch:  1100 Loss:  0.04651231002322778
Epoch:  1200 Loss:  0.02815121242923335
Epoch:  1300 Loss:  0.019605916607094816
Epoch:  1400 Loss:  0.014722898563264928
1.0


On further we can use optimizers(adam) and can get more acurate output prediction from the model .

-> optimizers are similiar to gradients descent but on using them they are more accurate then the  gradients

Test in test dataset

In [281]:
w1,b1,w2,b2 ,w3,b3= train(x_train,y_train, n_h1=9,n_h2=6, epochs=1500,learning_rate=0.01)
y_pred = predict(x_test,w1,b1,w2,b2,w3,b3)
y_pred = one_hot_encode(y_pred)
acuuracy = np.mean(y_pred == y_test)
print(acuuracy)

Epoch:  0 Loss:  1.0986133724740612
Epoch:  100 Loss:  1.0930769446469337
Epoch:  200 Loss:  1.0902479876536995
Epoch:  300 Loss:  1.0887936678428494
Epoch:  400 Loss:  1.0880110045284221
Epoch:  500 Loss:  1.087502678955219
Epoch:  600 Loss:  1.0869048424829657
Epoch:  700 Loss:  1.0851359830240765
Epoch:  800 Loss:  1.0735640938136002
Epoch:  900 Loss:  0.9661145052122528
Epoch:  1000 Loss:  0.5988469874799398
Epoch:  1100 Loss:  0.2650654566779675
Epoch:  1200 Loss:  0.077076548713589
Epoch:  1300 Loss:  0.04091687665234241
Epoch:  1400 Loss:  0.02697434643916605
1.0


In [282]:
print(y_train.shape)
print(y_test.shape)

(142, 3)
(36, 3)
